# jaxfne — 100k-neuron cortical column on Colab GPU/TPU

**Purpose:** get a REAL wall-clock timing receipt for `scripts/cortical_column_localized_workflow.py`'s
100k-neuron cortical column on actual GPU/TPU hardware, replacing the CPU-only estimate this
repo has carried until now (`plans.json` item `colab-notebook-gpu-tpu-100k-column`).

**Status: this notebook has NOT been run on GPU/TPU as of 2026-07-14.** It was authored and
structurally checked (imports, API calls) but the authoring environment has no GPU/TPU access —
run this in Google Colab yourself (Runtime → Change runtime type → GPU or TPU) and the printed
timing in the last cell will be a real, dated receipt, not an estimate.

**Known CPU-only baseline for comparison** (this repo's own prior measurement, same script,
same config): `construct()` ~216s, `simulate()` (100 steps) ~5s, N=100,000, ~10.0M edges
(`max_in_degree=100`, `spatial_sigma=0.15mm`), CPU-only.

**GPU/TPU speed estimate — a REASONED PRIOR, not a measurement, do not trust until this
notebook's own last cell has actually run:** the dominant per-step cost is `jax.ops.segment_sum`
over ~10M edges (a scatter-add pattern), not dense matmul. GPUs are historically well-suited to
scatter/gather workloads via flexible memory addressing; TPUs are optimized primarily for dense
matmul/convolution and are typically LESS efficient on scatter-add patterns than a comparable
GPU — so "TPU is always faster" is likely false for this specific workload; expect a CUDA GPU
(e.g. A100) to outperform a TPU here, not the reverse. Rough order-of-magnitude prior for a
GPU vs. the 527s CPU baseline for 10,000 steps (a different, longer run than the 100-step timing
above): JAX CPU→GPU speedups for segment_sum/scan-heavy sparse workloads of this size commonly
fall in the 10-50x range, which would put a GPU run at very roughly 10-50s for the same
1000ms/10,000-step `simulate()` — a genuine estimate range, not a benchmark. `construct()` itself
is a host-side/non-traced numpy loop by this repo's own design choice and would NOT speed up on
GPU/TPU, remaining ~216s regardless of accelerator — this notebook flags that explicitly so a
reader doesn't expect `construct()` to benefit from the runtime type.

## 1. Clone + install (Colab only — skip if already in a jaxfne checkout)

In [ ]:
import os
if not os.path.exists("jaxfne"):
    !git clone https://github.com/HNXJ/jaxfne.git
    %cd jaxfne
    !pip install -e ".[dev]" -q
else:
    %cd jaxfne


## 2. Detect the assigned accelerator\n\nColab's JAX auto-detects whatever accelerator the Runtime menu assigned — no manual `JAX_PLATFORMS` override needed once the runtime type is set to GPU or TPU in Colab's own Runtime menu (this cannot be set from code; do it via the UI before running this cell).

In [ ]:
import jax
print("jaxfne file:", __import__("jaxfne").__file__)  # confirm which checkout is actually imported
print("JAX devices:", jax.devices())
print("Default backend:", jax.default_backend())


## 3. Build the 100k-neuron column and time construct()/simulate() for real\n\nSame config as this repo's own CPU baseline (`scripts/cortical_column_localized_workflow.py`), just timed on whatever accelerator section 2 reported.

In [ ]:
import time
import sys
sys.path.insert(0, ".")
from scripts.cortical_column_localized_workflow import build_config
import jaxfne as jtfne

N = 100_000

t0 = time.time()
cfg = build_config(n=N)
model = jtfne.construct(cfg)
t_construct = time.time() - t0
print(f"construct() at N={N}: {t_construct:.1f}s (host-side, NOT accelerator-dependent by design)")

sim = jtfne.simulation(duration_ms=50.0, dt_ms=0.5, seed=0)  # 100 steps
t0 = time.time()
sig = model.simulate(sim)
sig.V_m.block_until_ready()  # force real completion before stopping the timer
t_simulate_100steps = time.time() - t0
print(f"simulate() 100 steps at N={N}: {t_simulate_100steps:.2f}s on {jax.default_backend()}")


## 4. Receipt\n\nRun this cell after section 3 completes — it prints the dated, real numbers this notebook exists to produce. Copy this output back into `plans.json`'s `colab-notebook-gpu-tpu-100k-column` item once run for real.

In [ ]:
import datetime
print("=== REAL RECEIPT (not an estimate) ===")
print("Date:", datetime.date.today().isoformat())
print("Backend:", jax.default_backend(), "| Devices:", jax.devices())
print(f"N={N} neurons")
print(f"construct(): {t_construct:.1f}s")
print(f"simulate() 100 steps: {t_simulate_100steps:.2f}s")
print(f"Finite check: {bool(jax.numpy.all(jax.numpy.isfinite(sig.V_m)))}")
